# Отчёт по лабораторной работе №2

## Spring MVC

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

## Цель работы

Получить практические навыки работы со Spring MVC. Разработать веб-приложение
для управления задачами и разделить его на модель, представление и контроллер.

## Задание

1. Изучить теоретический материал.
2. Создать список задач по примеру из методички.
3. Реализовать добавление задач, изменение статуса и удаление через веб-интерфейс.
4. Выполнить дополнительное задание и подготовить отчёт.

Выполнены дополнительные пункты 3, 5 и 6: подтверждение удаления,
поиск по задачам и оформление страницы.

## Ход работы

### 1. Создание и настройка проекта

Создан отдельный Gradle-проект `Demo` для второй лабораторной.
Использованы Java 17, Gradle 8.12 и Spring Boot 3.2.3.
Зависимость `spring-boot-starter-web` подключает Spring MVC и встроенный сервер,
а `spring-boot-starter-thymeleaf` используется для HTML-шаблонов.

Точка входа приложения находится в классе `DemoApplication`.

```groovy
plugins {
    id 'java'
    id 'org.springframework.boot' version '3.2.3'
    id 'io.spring.dependency-management' version '1.1.4'
}

group = 'ru.bsuedu.cad.lab'
version = '1.0'

java {
    sourceCompatibility = JavaVersion.VERSION_17
    targetCompatibility = JavaVersion.VERSION_17
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
}

tasks.withType(JavaCompile).configureEach {
    options.encoding = 'UTF-8'
    options.compilerArgs.add('-parameters')
}
```

```java
package ru.bsuedu.cad.lab;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class DemoApplication {

    public static void main(String[] args) {
        SpringApplication.run(DemoApplication.class, args);
    }
}
```

### 2. Модель задачи

Класс `Task` содержит идентификатор, название, описание и признак выполнения.
Новая задача создаётся со статусом «в работе». При изменении статуса
обновляется поле `completed`.

```java
package ru.bsuedu.cad.lab;

public class Task {

    private final Long id;
    private final String title;
    private final String description;
    private boolean completed;

    public Task(Long id, String title, String description) {
        this.id = id;
        this.title = title;
        this.description = description;
    }

    public Long getId() {
        return id;
    }

    public String getTitle() {
        return title;
    }

    public String getDescription() {
        return description;
    }

    public boolean isCompleted() {
        return completed;
    }

    public void setCompleted(boolean completed) {
        this.completed = completed;
    }
}
```

### 3. Хранение задач

Класс `TaskRepository` хранит задачи в памяти приложения.
Для каждой новой задачи создаётся уникальный идентификатор.
Репозиторий добавляет задачи, возвращает список, обновляет статус и удаляет задачи.
После остановки приложения список очищается.

```java
package ru.bsuedu.cad.lab;

import org.springframework.stereotype.Repository;

import java.util.ArrayList;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;

@Repository
public class TaskRepository {

    private final Map<Long, Task> tasks = new LinkedHashMap<>();
    private long nextId = 1;

    public synchronized void addTask(String title, String description) {
        Task task = new Task(nextId++, title, description);
        tasks.put(task.getId(), task);
    }

    public synchronized List<Task> getAllTasks() {
        return new ArrayList<>(tasks.values());
    }

    public synchronized boolean updateTask(Long id, boolean completed) {
        Task task = tasks.get(id);
        if (task == null) {
            return false;
        }
        task.setCompleted(completed);
        return true;
    }

    public synchronized boolean deleteTask(Long id) {
        return tasks.remove(id) != null;
    }
}
```

### 4. Сервисный слой

`TaskService` содержит логику работы с задачами и обращается к репозиторию.
При добавлении удаляются пробелы по краям текста и проверяется название.
Поиск выполняется по названию и описанию без учёта регистра.
Если задача с указанным идентификатором отсутствует, возвращается ответ 404.

```java
package ru.bsuedu.cad.lab;

import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Service;
import org.springframework.web.server.ResponseStatusException;

import java.util.List;
import java.util.Locale;

@Service
public class TaskService {

    private final TaskRepository taskRepository;

    public TaskService(TaskRepository taskRepository) {
        this.taskRepository = taskRepository;
    }

    public void addTask(String title, String description) {
        String cleanTitle = title.strip();
        String cleanDescription = description.strip();
        if (cleanTitle.isEmpty()) {
            throw new IllegalArgumentException("введите название задачи");
        }
        if (cleanTitle.length() > 100 || cleanDescription.length() > 1000) {
            throw new IllegalArgumentException("название до 100 символов, описание до 1000");
        }
        taskRepository.addTask(cleanTitle, cleanDescription);
    }

    public List<Task> getTasks(String query) {
        String search = query.strip().toLowerCase(Locale.ROOT);
        return taskRepository.getAllTasks().stream()
                .filter(task -> task.getTitle().toLowerCase(Locale.ROOT).contains(search)
                        || task.getDescription().toLowerCase(Locale.ROOT).contains(search))
                .toList();
    }

    public void updateTask(Long id, boolean completed) {
        if (!taskRepository.updateTask(id, completed)) {
            throw new ResponseStatusException(HttpStatus.NOT_FOUND, "задача не найдена");
        }
    }

    public void deleteTask(Long id) {
        if (!taskRepository.deleteTask(id)) {
            throw new ResponseStatusException(HttpStatus.NOT_FOUND, "задача не найдена");
        }
    }
}
```

### 5. Контроллер

`TaskController` принимает HTTP-запросы и вызывает сервис.
Метод `index` добавляет список задач в `Model` и возвращает имя шаблона `index`.
После добавления, смены статуса или удаления выполняется переход на главную страницу.

| Метод | Адрес | Действие |
| --- | --- | --- |
| GET | `/` | Показать список задач |
| GET | `/?q=текст` | Найти задачи |
| POST | `/addTask` | Добавить задачу |
| POST | `/updateTask/{id}` | Изменить статус |
| POST | `/deleteTask/{id}` | Удалить задачу |

```java
package ru.bsuedu.cad.lab;

import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PathVariable;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.RequestParam;
import org.springframework.web.servlet.mvc.support.RedirectAttributes;

@Controller
public class TaskController {

    private final TaskService taskService;

    public TaskController(TaskService taskService) {
        this.taskService = taskService;
    }

    @GetMapping("/")
    public String index(@RequestParam(defaultValue = "") String q, Model model) {
        model.addAttribute("tasks", taskService.getTasks(q));
        model.addAttribute("query", q.strip());
        return "index";
    }

    @PostMapping("/addTask")
    public String addTask(@RequestParam String title,
                          @RequestParam(defaultValue = "") String description,
                          RedirectAttributes attributes) {
        try {
            taskService.addTask(title, description);
            attributes.addFlashAttribute("message", "задача добавлена");
        } catch (IllegalArgumentException exception) {
            attributes.addFlashAttribute("error", exception.getMessage());
        }
        return "redirect:/";
    }

    @PostMapping("/updateTask/{id}")
    public String updateTask(@PathVariable Long id, @RequestParam boolean completed,
                             RedirectAttributes attributes) {
        taskService.updateTask(id, completed);
        attributes.addFlashAttribute("message", "статус изменён");
        return "redirect:/";
    }

    @PostMapping("/deleteTask/{id}")
    public String deleteTask(@PathVariable Long id, RedirectAttributes attributes) {
        taskService.deleteTask(id);
        attributes.addFlashAttribute("message", "задача удалена");
        return "redirect:/";
    }
}
```

### 6. Представление Thymeleaf

Шаблон `index.html` расположен в `src/main/resources/templates`.
Он показывает задачи, форму добавления, поле поиска и кнопки управления.
Атрибут `th:each` перебирает задачи, `th:text` выводит данные,
а `th:action` задаёт адрес отправки формы.

Названия, описания и статусы отображаются на русском языке.

```html
<!DOCTYPE html>
<html lang="ru" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>список задач</title>
    <link rel="stylesheet" th:href="@{/style.css}">
    <script th:src="@{/delete-confirm.js}" defer></script>
</head>
<body>
<main class="page">
    <header class="page-header">
        <div class="app-icon" aria-hidden="true">✓</div>
        <div>
            <h1>список задач</h1>
            <p>добавляйте задачи и отмечайте выполненные</p>
        </div>
    </header>

    <div class="notice success" role="status" th:if="${message}" th:text="${message}"></div>
    <div class="notice error" role="alert" th:if="${error}" th:text="${error}"></div>

    <div class="layout">
        <section class="panel tasks-panel" aria-labelledby="tasks-heading">
            <div class="section-header">
                <h2 id="tasks-heading">мои задачи</h2>
                <span class="count" th:text="${tasks.size()}">0</span>
            </div>

            <form class="search" th:action="@{/}" method="get">
                <label class="visually-hidden" for="query">поиск по задачам</label>
                <input id="query" name="q" type="search" th:value="${query}"
                       placeholder="найти по названию или описанию">
                <button type="submit" class="button secondary">найти</button>
                <a class="reset" th:if="${!query.isEmpty()}" th:href="@{/}">сбросить</a>
            </form>

            <div class="empty" th:if="${tasks.isEmpty()}">
                <div class="empty-icon" aria-hidden="true">☰</div>
                <h3 th:text="${query.isEmpty() ? 'задач пока нет' : 'ничего не найдено'}"></h3>
                <p th:text="${query.isEmpty() ? 'добавьте первую задачу с помощью формы' : 'попробуйте изменить запрос'}"></p>
            </div>

            <ul class="task-list" th:if="${!tasks.isEmpty()}">
                <li class="task" th:each="task : ${tasks}" th:classappend="${task.completed ? 'done' : ''}">
                    <div class="task-top">
                        <h3 th:text="${task.title}"></h3>
                        <span class="status" th:classappend="${task.completed ? 'completed' : 'pending'}"
                              th:text="${task.completed ? 'выполнена' : 'в работе'}"></span>
                    </div>
                    <p class="task-description" th:if="${!task.description.isEmpty()}"
                       th:text="${task.description}"></p>
                    <div class="task-actions">
                        <form th:action="@{/updateTask/{id}(id=${task.id})}" method="post">
                            <input type="hidden" name="completed" th:value="${!task.completed}">
                            <button type="submit" class="button secondary"
                                    th:text="${task.completed ? 'вернуть в работу' : 'выполнить'}"></button>
                        </form>
                        <form th:action="@{/deleteTask/{id}(id=${task.id})}" method="post"
                              data-delete-form th:attr="data-task-title=${task.title}">
                            <button type="submit" class="button danger"
                                    th:aria-label="${'удалить задачу ' + task.title}">удалить</button>
                        </form>
                    </div>
                </li>
            </ul>
        </section>

        <aside class="panel add-panel" aria-labelledby="add-heading">
            <h2 id="add-heading">новая задача</h2>
            <p class="form-hint">что нужно сделать</p>
            <form th:action="@{/addTask}" method="post" class="add-form">
                <label for="title">название</label>
                <input id="title" name="title" type="text" maxlength="100" required
                       placeholder="например, подготовить отчёт">
                <label for="description">описание</label>
                <textarea id="description" name="description" rows="4" maxlength="1000"
                          placeholder="детали задачи"></textarea>
                <button type="submit" class="button primary">добавить задачу</button>
            </form>
            <p class="storage-note">задачи хранятся до остановки приложения</p>
        </aside>
    </div>
</main>
<dialog id="delete-dialog" aria-labelledby="delete-heading">
    <form method="dialog">
        <h2 id="delete-heading">удалить задачу</h2>
        <p id="delete-title"></p>
        <p class="dialog-hint">задача будет удалена из списка</p>
        <div class="dialog-actions">
            <button class="button secondary" value="cancel" autofocus>оставить</button>
            <button class="button delete-confirm" value="delete">удалить</button>
        </div>
    </form>
</dialog>
</body>
</html>
```

### 7. Проверка работы приложения

Проект собран командой `gradlew.bat bootJar` и запущен локально.
Приложение открыто в браузере по адресу `http://localhost:8081`.

**Пустой список**

При первом открытии задач нет. На странице доступна форма добавления.

[![Пустой список задач](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/empty.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/empty.jpg)

**Добавление задач**

Через форму добавлены задачи «подготовить отчёт» и «прочитать лекцию»
с описаниями. Обе задачи появились в списке со статусом «в работе».

[![Добавленные задачи](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/tasks.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/tasks.jpg)

**Изменение статуса**

Для первой задачи нажата кнопка «выполнить».
Статус изменился на «выполнена», появилась кнопка «вернуть в работу».
Обратное переключение статуса также проверено.

[![Задача отмечена выполненной](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/completed.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/completed.jpg)

**Удаление задачи**

После подтверждения удаления задача «прочитать лекцию» исчезла из списка.
На странице осталось одно задание и появилось сообщение «задача удалена».

[![Список после удаления задачи](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/deleted.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/deleted.jpg)

**Проверка названия**

При отправке названия из пробелов задача не добавилась.
На странице появилось сообщение «введите название задачи».
Существующие задачи сохранились.

[![Проверка пустого названия](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/validation.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/validation.jpg)

### 8. Дополнительные задания

**Пункт 3. Подтверждение удаления**

Перед удалением открывается окно с названием задачи и кнопками «оставить» и «удалить».
Кнопка «оставить» закрывает окно и сохраняет задачу.
Удаление выполняется только после подтверждения.
Проверены оба действия.

Обработчики находятся в файле `delete-confirm.js`.

```javascript
const deleteDialog = document.getElementById('delete-dialog');
const deleteTitle = document.getElementById('delete-title');
let selectedForm = null;

document.querySelectorAll('[data-delete-form]').forEach(form => {
    form.addEventListener('submit', event => {
        event.preventDefault();
        selectedForm = form;
        deleteTitle.textContent = form.dataset.taskTitle;
        deleteDialog.returnValue = '';
        deleteDialog.showModal();
    });
});

deleteDialog.addEventListener('close', () => {
    if (deleteDialog.returnValue === 'delete' && selectedForm) {
        selectedForm.submit();
    }
    selectedForm = null;
});
```

[![Подтверждение удаления задачи](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/delete-confirm.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/delete-confirm.jpg)

**Пункт 5. Поиск по задачам**

Запрос «ОТЧЁТ» нашёл задачу «подготовить отчёт», хотя регистр букв различается.
Также проверен поиск по слову «SPRING» из описания.
Кнопка «сбросить» возвращает полный список.

[![Поиск по названию без учёта регистра](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/search.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/search.jpg)

По запросу «покупки» совпадений нет. На странице отображается сообщение
«ничего не найдено», сами задачи при этом не удаляются.

[![Поиск без совпадений](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/search-empty.jpg)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab2/images/search-empty.jpg)

**Пункт 6. Оформление страницы**

Для страницы создан файл `style.css`. Форма добавления и список расположены
в отдельных блоках. Статусы выделены цветом, выполненная задача зачёркивается.
На узком экране блоки располагаются в один столбец.

```css
:root {
    font-family: "Segoe UI", Arial, sans-serif;
    color: #25334b;
    background: #f3f5f9;
    font-synthesis: none;
}

* { box-sizing: border-box; }
body { margin: 0; }
button, input, textarea { font: inherit; }
button, a { -webkit-tap-highlight-color: transparent; }
.page { max-width: 1160px; margin: 0 auto; padding: 48px 28px; }
.page-header { display: flex; align-items: center; gap: 18px; margin-bottom: 30px; }
.app-icon {
    width: 54px; height: 54px; border-radius: 16px; display: grid;
    place-items: center; background: #355fd1; color: white; font-size: 30px;
}
h1, h2, h3, p { margin: 0; }
h1 { font-size: 30px; font-weight: 650; letter-spacing: -0.6px; }
.page-header p { color: #6b778d; margin-top: 5px; font-size: 15px; }
h2 { font-size: 19px; font-weight: 650; }
.layout { display: grid; grid-template-columns: minmax(0, 1fr) 330px; gap: 24px; align-items: start; }
.panel { background: white; border: 1px solid #e2e7ef; border-radius: 16px; padding: 25px; }
.section-header { display: flex; align-items: center; gap: 10px; margin-bottom: 20px; }
.count { background: #eef2fc; color: #355fd1; border-radius: 7px; padding: 3px 9px; font-size: 13px; }
```

## Вывод

В ходе лабораторной работы разработан список задач на Spring MVC и Thymeleaf.
Созданы модель, репозиторий, сервис и контроллер.
Реализованы добавление задач, переключение статуса и удаление.
Выполнены дополнительные задания: подтверждение удаления, поиск и оформление страницы.
Работа приложения проверена через веб-интерфейс.